In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def get_sampling_discrepancy_analytic(logits_ref, logits_score, labels):
    assert logits_ref.shape[0] == 1
    assert logits_score.shape[0] == 1
    assert labels.shape[0] == 1

    if logits_ref.size(-1) != logits_score.size(-1):
        vocab_size = min(logits_ref.size(-1), logits_score.size(-1))
        logits_ref = logits_ref[:, :, :vocab_size]
        logits_score = logits_score[:, :, :vocab_size]

    labels = labels.unsqueeze(-1) if labels.ndim == logits_score.ndim - 1 else labels
    lprobs_score = torch.log_softmax(logits_score, dim=-1)
    probs_ref = torch.softmax(logits_ref, dim=-1)
    log_likelihood = lprobs_score.gather(dim=-1, index=labels).squeeze(-1)
    mean_ref = (probs_ref * lprobs_score).sum(dim=-1)
    var_ref = (probs_ref * torch.square(lprobs_score)).sum(dim=-1) - torch.square(mean_ref)
    discrepancy = (log_likelihood.sum(dim=-1) - mean_ref.sum(dim=-1)) / var_ref.sum(dim=-1).sqrt()
    return discrepancy.mean().item()

def load_model(model_fullname, device):
    model = AutoModelForCausalLM.from_pretrained(
        model_fullname,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model.eval()
    return model

def load_tokenizer(model_fullname):
    tokenizer = AutoTokenizer.from_pretrained(model_fullname, padding_side='right')
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    return tokenizer

class FastDetectGPT:
    def __init__(self, args):
        self.args = args
        self.criterion_fn = get_sampling_discrepancy_analytic
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Scoring model
        self.scoring_tokenizer = load_tokenizer(args["scoring_model_name"])
        self.scoring_model = load_model(args["scoring_model_name"], self.device)

        # Sampling model
        if args["sampling_model_name"] != args["scoring_model_name"]:
            self.sampling_tokenizer = load_tokenizer(args["sampling_model_name"])
            self.sampling_model = load_model(args["sampling_model_name"], self.device)
        else:
            self.sampling_tokenizer = self.scoring_tokenizer
            self.sampling_model = self.scoring_model

    # Compute discrepancy for a single text (preserves original behavior)
    def compute_crit(self, text):
        # Tokenize scoring text
        scoring_inputs = self.scoring_tokenizer(
            text, truncation=True, return_tensors="pt", padding=True
        )
        scoring_inputs = {k: v.to(self.device) for k, v in scoring_inputs.items()}
        labels = scoring_inputs['input_ids'][:, 1:]

        with torch.inference_mode():
            logits_score = self.scoring_model(**scoring_inputs).logits[:, :-1]

            if self.sampling_model is self.scoring_model:
                logits_ref = logits_score
            else:
                sampling_inputs = self.sampling_tokenizer(
                    text, truncation=True, return_tensors="pt", padding=True
                )
                sampling_inputs = {k: v.to(self.device) for k, v in sampling_inputs.items()}
                # Align labels with sampling model
                assert torch.all(sampling_inputs['input_ids'][:, 1:] == labels)
                logits_ref = self.sampling_model(**sampling_inputs).logits[:, :-1]

            score = self.criterion_fn(logits_ref, logits_score, labels)

        return score

    def compute_crit_batch(self, texts):
        device = self.device
    
        # Tokenize all texts at once
        scoring_inputs = self.scoring_tokenizer(
            texts, truncation=True, return_tensors="pt", padding=True
        )
        scoring_inputs = {k: v.to(device) for k, v in scoring_inputs.items()}
        labels = scoring_inputs['input_ids'][:, 1:]
        seq_lengths = (scoring_inputs['attention_mask'][:, 1:] > 0).sum(dim=1)  # exclude padding
    
        with torch.inference_mode():
            logits_score = self.scoring_model(**scoring_inputs).logits[:, :-1]
    
            if self.sampling_model is self.scoring_model:
                logits_ref = logits_score
            else:
                sampling_inputs = self.sampling_tokenizer(
                    texts, truncation=True, return_tensors="pt", padding=True
                )
                sampling_inputs = {k: v.to(device) for k, v in sampling_inputs.items()}
                logits_ref = self.sampling_model(**sampling_inputs).logits[:, :-1]
    
            # Compute discrepancy per sequence, slicing to real length
            scores = []
            for i in range(len(texts)):
                length = seq_lengths[i]
                score = self.criterion_fn(
                    logits_ref[i:i+1, :length, :],
                    logits_score[i:i+1, :length, :],
                    labels[i:i+1, :length]
                )
                scores.append(score)

        return scores



In [ ]:
args = {
    "scoring_model_name": 'Qwen/Qwen2.5-7B-Instruct',
    "sampling_model_name": 'Qwen/Qwen2.5-7B',
    # "device": "cuda",
}

text = "This is a sample text to evaluate."
detector = FastDetectGPT(args)
labels = detector.compute_crit(text)


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from dotenv import load_dotenv
load_dotenv()

import json
from tqdm import tqdm
import pandas as pd
data_path = "../../../data/external_data/fox8_23_dataset.ndjson"
with open(data_path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f]

fox8_23_dicts = []
for sample in tqdm(data):
    label = 0 if sample["label"] == "human" else 1
    user_id = sample["user_id"]
    for tweet in sample["user_tweets"]:
        fox8_23_dicts.append({
            "text": tweet["text"],
            "user_id": user_id,
            "label": label
        })
fox8_23 = pd.DataFrame(fox8_23_dicts)

In [ ]:
from tqdm.auto import tqdm
import torch
import gc

batch_size = 128

score = []
for start in tqdm(range(0, len(fox8_23), batch_size), desc="Progress"):
    end = start + batch_size
    batch_text = fox8_23["text"].iloc[start:end].tolist()

    # Compute scores in batch
    score.extend(detector.compute_crit_batch(batch_text))

    del batch_text
    torch.cuda.empty_cache()
    gc.collect()

    if len(score) % 10000 == 0:
      data_sample = fox8_23.iloc[:len(score)]
      data_sample["fd_score"] = score
      data_sample.to_parquet("fox8_fd_scores.parquet")

fox8_23["fd_score"] = score
fox8_23.to_parquet("fox8_fd_scores.parquet")